# Kaggle コミットで Hardcore 追加学習を回すテンプレ

Basic で学習済みのベストモデルを W&B から取得し、Hardcore（`BipedalWalkerHardcore-v3`）へ追加学習（カリキュラム学習）するノートです。sweep 用の `kaggle_commit.ipynb` と同じ準備手順（clone → pip install → W&B ログイン）で、最後のセルだけが違います。

**注意事項（読んでから実行）**

- このノートは **CPU セッション** で使う。GPU は選択しない。
- **インターネット接続を有効化** すること（Settings → Internet を On。電話番号認証が必要）。
- W&B の API キーは Add-ons → Secrets に `WANDB_API_KEY` として登録し、チェックボックスを On にする。
- **編集するのは最後のセルの変数 3 つだけ**（`RESUME_RUN_PATH` / `SEED` / `TOTAL_TIMESTEPS`）。
- 実測で 100万ステップ ≒ 8〜10 時間。セッション上限（約12時間）に収まるが、**途中で切られると checkpoint も消える**ので、心配なら `TOTAL_TIMESTEPS` を減らして複数セッションに分ける。
- 使い方: 右上の Save Version → 「**Save & Run All（Commit）**」で放置実行する。
- 学習が終わると、ベストモデルの走りの動画が **自動で W&B の run にアップロードされる**（train.py の機能。追加操作は不要）。


In [ ]:
# 1. リポジトリを clone
# 【なぜ】Kaggle が用意するマシンはまっさらで、私たちのコードが入っていない。
#         まず GitHub からコード一式をこのマシンにダウンロードする。
# 【成功すると】SB3-Team フォルダができて src/train.py などが使える状態になり、
#              %cd でそのフォルダに移動する。
!git clone https://github.com/shironaganegi/SB3-Team.git
%cd SB3-Team

In [ ]:
# 2. 依存ライブラリをインストール
# 【なぜ】学習コードは PyTorch・gymnasium・sb3-contrib などに依存しているが、
#         まっさらなマシンには入っていない（または版が合わない）ため。
!pip install -r requirements.txt

In [ ]:
# 3. W&B にログイン
# 【なぜ】このマシンから W&B のモデルファイルを取得し、学習結果を送るため。
#         キーは Add-ons → Secrets の WANDB_API_KEY から読む（直書き禁止）。
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")
!wandb login --relogin $WANDB_API_KEY

In [ ]:
# 4. ベースモデルを W&B から取得し、Hardcore へ追加学習（このノートの本体）
# 【なぜ】Hardcore をゼロから学習するのは CPU ではほぼ無理なので、
#         「歩ける」Basic モデルに障害物対応だけを追加学習させる（README §7 Step 4a）。
#         ベースモデルは W&B の run に model.zip として保存されているので、
#         run のパスを指定してダウンロードする。
# 【成功すると】W&B に hardcore の run が1本増え、学習が終わると
#              モデル(zip)が run の Files にアップロードされる。
#
# 【書き換える場所】下の3つの変数だけ。
#   RESUME_RUN_PATH … ベースにする run のパス（entity/project/run_id）。
#                     既定値は Hardcore 追加学習1回目の成果（chocolate-yogurt-12:
#                     eval報酬160・採点再現は完走0/5・reward 61.2）。ここからの続行が2周目。
#                     Basic ベストからやり直す場合は sai3desuyo-/bipedal-timetrial/xsrplaip
#                     （drawn-sweep-5: eval報酬336・lr4.43e-4・seed1）に戻す。
#   SEED            … 乱数シード。人ごとに変えて分担すると探索が広がる。
#   TOTAL_TIMESTEPS … 追加学習のステップ数。100万 ≒ 8〜10時間（12時間上限に注意）。

RESUME_RUN_PATH = "sai3desuyo-/bipedal-timetrial/an3wpjb5"
SEED = 0
TOTAL_TIMESTEPS = 1_000_000

import wandb, yaml

# --- ベースモデルのダウンロード ---
api = wandb.Api()
base_run = api.run(RESUME_RUN_PATH)
model_file = [f for f in base_run.files() if f.name == "model.zip"]
assert model_file, f"{RESUME_RUN_PATH} に model.zip がありません（Finished した run を指定してください）"
model_file[0].download(root="base_model", replace=True)
print(f"[ok] base_model/model.zip を取得（{base_run.name}: eval_reward={base_run.summary.get('eval/mean_reward')}）")

# --- config のコピーを作る（use_wandb を有効化し、このセルの変数を反映）---
cfg = yaml.safe_load(open("configs/hardcore_finetune.yaml"))
cfg["use_wandb"] = True
cfg["seed"] = SEED
cfg["total_timesteps"] = TOTAL_TIMESTEPS
yaml.safe_dump(cfg, open("hardcore_kaggle.yaml", "w"), allow_unicode=True)

# --- 追加学習を実行 ---
!python src/train.py --config hardcore_kaggle.yaml --resume base_model/model.zip